# 📥 Téléchargement Étendu des Données Historiques

Ce notebook télécharge **TOUTES** les données historiques nécessaires pour le projet de prédiction électorale 2027.

## 🎯 Données téléchargées

### Phase 1 - Référentiels (✅ Déjà fait partiellement)
- Référentiel communes 2024 ✅
- Référentiels communes 1999-2023 🔄

### Phase 2 - Démographie (⭐ Priorité haute)
1. Population historique 1876-2023 (6.7 MB)
2. Naissances 2008-2024 (24 MB)
3. Décès 2008-2024 (24 MB)

### Phase 3 - Économie (⭐⭐ Priorité moyenne)
4. CSP actifs 1968-2022 (28.5 MB)
5. Secteur activité 1968-2022 (23.5 MB)
6. Comptes communes 2000-2022 (~1-2 GB par tranches)

### Phase 4 - Éducation (⭐ Priorité basse)
7. Diplômes 2022 (81 MB)
8. CSP × Diplôme 1968-2022 (51.9 MB)

**Volume total estimé** : ~3.5 GB (compressé) → ~5 GB (décompressé)

## ⚙️ Configuration

In [1]:
# Configuration inline (compatible VS Code Jupyter)
from pathlib import Path
import requests
from zipfile import ZipFile
from io import BytesIO
import time

# Import tqdm avec fallback si non disponible
try:
    from tqdm.notebook import tqdm
    HAS_TQDM = True
except ImportError:
    print("⚠️  tqdm non installé - pas de barres de progression")
    print("   Pour l'installer: pip install tqdm\n")
    HAS_TQDM = False
    # Mock tqdm
    class tqdm:
        def __init__(self, *args, **kwargs):
            pass
        def __enter__(self):
            return self
        def __exit__(self, *args):
            pass
        def update(self, n):
            pass

# Détection automatique du répertoire projet
notebook_dir = Path.cwd()
PROJECT_ROOT = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir

# Chemins Medallion Architecture
DATA_BRONZE = PROJECT_ROOT / 'data' / 'bronze'
DATA_SILVER = PROJECT_ROOT / 'data' / 'silver'
DATA_GOLD = PROJECT_ROOT / 'data' / 'gold'

# Créer les dossiers si nécessaire
DATA_BRONZE.mkdir(parents=True, exist_ok=True)

print(f"✅ Projet : {PROJECT_ROOT}")
print(f"✅ Bronze : {DATA_BRONZE}")
print(f"✅ Espace disque disponible : {sum(f.stat().st_size for f in DATA_BRONZE.rglob('*') if f.is_file()) / 1e9:.2f} GB")

✅ Projet : /app
✅ Bronze : /app/data/bronze
✅ Espace disque disponible : 2.57 GB


## 🔧 Fonctions Utilitaires

In [2]:
def download_file(url, destination, description="Téléchargement", force=False):
    """
    Télécharge un fichier avec barre de progression.
    
    Args:
        url: URL du fichier
        destination: Chemin de destination
        description: Description pour la barre de progression
        force: Forcer le téléchargement même si le fichier existe
    
    Returns:
        bool: True si succès, False sinon
    """
    destination = Path(destination)
    
    # Vérifier si le fichier existe déjà
    if destination.exists() and not force:
        size_mb = destination.stat().st_size / 1e6
        print(f"⏭️  {destination.name} existe déjà ({size_mb:.2f} MB) - SKIP")
        return True
    
    try:
        print(f"📥 {description}... ", end='', flush=True)
        # Télécharger avec streaming
        response = requests.get(url, stream=True, timeout=120)
        response.raise_for_status()
        
        # Taille totale
        total_size = int(response.headers.get('content-length', 0))
        downloaded = 0
        
        # Téléchargement avec ou sans tqdm
        if HAS_TQDM:
            with open(destination, 'wb') as f, tqdm(
                desc=description,
                total=total_size,
                unit='B',
                unit_scale=True,
                unit_divisor=1024,
            ) as pbar:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        pbar.update(len(chunk))
        else:
            # Sans tqdm - affichage simple du pourcentage
            with open(destination, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
                        downloaded += len(chunk)
                        if total_size > 0:
                            percent = (downloaded / total_size) * 100
                            print(f"\r📥 {description}... {percent:.1f}%", end='', flush=True)
        
        size_mb = destination.stat().st_size / 1e6
        print(f"\r✅ {destination.name} téléchargé ({size_mb:.2f} MB)" + " " * 20)
        return True
        
    except Exception as e:
        print(f"\r❌ Erreur: {e}" + " " * 20)
        if destination.exists():
            destination.unlink()  # Supprimer fichier partiel
        return False


def download_and_extract_zip(url, extract_to, description="Téléchargement ZIP", force=False):
    """
    Télécharge et extrait un fichier ZIP.
    
    Args:
        url: URL du fichier ZIP
        extract_to: Dossier de destination
        description: Description pour la barre de progression
        force: Forcer le téléchargement même si les fichiers existent
    
    Returns:
        bool: True si succès, False sinon
    """
    extract_to = Path(extract_to)
    extract_to.mkdir(parents=True, exist_ok=True)
    
    # Vérifier si déjà extrait (au moins 1 fichier dans le dossier)
    if not force and any(extract_to.iterdir()):
        print(f"⏭️  {extract_to.name} déjà extrait - SKIP")
        return True
    
    try:
        print(f"📥 {description}...")
        response = requests.get(url, timeout=60)
        response.raise_for_status()
        
        # Extraire le ZIP en mémoire
        with ZipFile(BytesIO(response.content)) as zip_file:
            zip_file.extractall(extract_to)
        
        files_count = len(list(extract_to.rglob('*')))
        print(f"✅ {files_count} fichiers extraits dans {extract_to.name}")
        return True
        
    except Exception as e:
        print(f"❌ Erreur: {e}")
        return False


def download_cog_millesimes(years, destination_folder, force=False):
    """
    Télécharge les référentiels COG pour plusieurs millésimes.
    
    Args:
        years: Liste des années (ex: range(1999, 2024))
        destination_folder: Dossier de destination
        force: Forcer le téléchargement
    
    Returns:
        dict: {année: True/False} pour succès/échec
    """
    destination_folder = Path(destination_folder)
    destination_folder.mkdir(parents=True, exist_ok=True)
    
    # URLs COG selon l'année (format varie selon millésime)
    results = {}
    
    for year in years:
        # Format URL change selon l'année
        if year >= 2019:
            # Nouveau format depuis 2019
            if year == 2025:
                base_url = "https://www.insee.fr/fr/statistiques/fichier/8377162"
            elif year == 2024:
                base_url = "https://www.insee.fr/fr/statistiques/fichier/7766585"
            elif year == 2023:
                base_url = "https://www.insee.fr/fr/statistiques/fichier/6800675"
            elif year == 2022:
                base_url = "https://www.insee.fr/fr/statistiques/fichier/6051727"
            elif year == 2021:
                base_url = "https://www.insee.fr/fr/statistiques/fichier/5057840"
            elif year == 2020:
                base_url = "https://www.insee.fr/fr/statistiques/fichier/4316069"
            elif year == 2019:
                base_url = "https://www.insee.fr/fr/statistiques/fichier/3720946"
            
            url = f"{base_url}/v_commune_{year}.csv"
        else:
            # Ancien format 1999-2018
            url_ids = {
                2018: "3363419", 2017: "2666684", 2016: "2114819", 2015: "2560698",
                2014: "2560563", 2013: "2560615", 2012: "2560620", 2011: "2560625",
                2010: "2560630", 2009: "2560635", 2008: "2560640", 2007: "2560646",
                2006: "2560651", 2005: "2560656", 2004: "2560661", 2003: "2560666",
                2002: "2560671", 2001: "2560676", 2000: "2560681", 1999: "2560686"
            }
            url_id = url_ids.get(year)
            if not url_id:
                print(f"❌ Année {year} non supportée")
                results[year] = False
                continue
            
            url = f"https://www.insee.fr/fr/statistiques/fichier/{url_id}/comsimp{year}.zip"
        
        destination = destination_folder / f"referentiel_communes_{year}.csv"
        
        # Télécharger
        if year >= 2019:
            # CSV direct
            results[year] = download_file(url, destination, f"COG {year}", force)
        else:
            # ZIP à extraire
            temp_folder = destination_folder / f"temp_cog_{year}"
            if download_and_extract_zip(url, temp_folder, f"COG {year}", force):
                # Trouver le CSV dans le ZIP
                csv_files = list(temp_folder.glob('*.csv'))
                if csv_files:
                    csv_files[0].rename(destination)
                    # Supprimer le dossier temporaire
                    import shutil
                    shutil.rmtree(temp_folder)
                    results[year] = True
                else:
                    print(f"❌ Aucun CSV trouvé dans le ZIP {year}")
                    results[year] = False
            else:
                results[year] = False
        
        time.sleep(0.5)  # Pause pour ne pas surcharger le serveur INSEE
    
    # Résumé
    success_count = sum(1 for v in results.values() if v)
    print(f"\n📊 Résumé COG : {success_count}/{len(results)} millésimes téléchargés")
    
    return results

print("✅ Fonctions utilitaires chargées")

✅ Fonctions utilitaires chargées


## 📋 Phase 1 - Référentiels COG (1999-2024)

Télécharge tous les millésimes du Code Officiel Géographique.
**Essentiel** pour faire des jointures historiques avec les données INSEE.

In [3]:
print("🚀 Phase 1 - Référentiels COG (1999-2024)\n")

cog_folder = DATA_BRONZE / 'referentiels_cog'
cog_folder.mkdir(parents=True, exist_ok=True)

# Télécharger tous les millésimes de 1999 à 2024
cog_results = download_cog_millesimes(
    years=range(1999, 2025),  # 1999 à 2024 inclus
    destination_folder=cog_folder,
    force=False  # Ne pas retélécharger si déjà présent
)

print("\n✅ Phase 1 terminée")

🚀 Phase 1 - Référentiels COG (1999-2024)

📥 COG 1999...
❌ Erreur: 500 Server Error:  for url: https://www.insee.fr/fr/statistiques/fichier/2560686/comsimp1999.zip
📥 COG 2000...
❌ Erreur: 500 Server Error:  for url: https://www.insee.fr/fr/statistiques/fichier/2560681/comsimp2000.zip
📥 COG 2001...
❌ Erreur: 500 Server Error:  for url: https://www.insee.fr/fr/statistiques/fichier/2560676/comsimp2001.zip
📥 COG 2002...
❌ Erreur: 500 Server Error:  for url: https://www.insee.fr/fr/statistiques/fichier/2560671/comsimp2002.zip
📥 COG 2003...
❌ Erreur: 500 Server Error:  for url: https://www.insee.fr/fr/statistiques/fichier/2560666/comsimp2003.zip
📥 COG 2004...
❌ Erreur: 500 Server Error:  for url: https://www.insee.fr/fr/statistiques/fichier/2560661/comsimp2004.zip
📥 COG 2005...
❌ Erreur: 500 Server Error:  for url: https://www.insee.fr/fr/statistiques/fichier/2560656/comsimp2005.zip
📥 COG 2006...
❌ Erreur: 500 Server Error:  for url: https://www.insee.fr/fr/statistiques/fichier/2560651/comsim

## 👥 Phase 2 - Démographie Historique

### 2.1 - Population 1876-2023 (6.7 MB)
147 années de données démographiques !

In [4]:
print("🚀 Phase 2.1 - Population historique 1876-2023\n")

url_population = "https://www.insee.fr/fr/statistiques/fichier/1893198/base-pop-historiques-1876-2023.xlsx"
destination_pop = DATA_BRONZE / "population_historique_1876_2023.xlsx"

download_file(url_population, destination_pop, "Population 1876-2023")

print("\n✅ Phase 2.1 terminée")

🚀 Phase 2.1 - Population historique 1876-2023

❌ Erreur: 500 Server Error:  for url: https://www.insee.fr/fr/statistiques/fichier/1893198/base-pop-historiques-1876-2023.xlsx                    

✅ Phase 2.1 terminée


### 2.2 - Naissances 2008-2024 (24 MB)

In [5]:
print("🚀 Phase 2.2 - Naissances 2008-2024\n")

url_naissances = "https://www.insee.fr/fr/statistiques/fichier/1893255/DS_ETAT_CIVIL_NAIS_COMMUNES.zip"
destination_naissances = DATA_BRONZE / "naissances_2008_2024"

download_and_extract_zip(url_naissances, destination_naissances, "Naissances 2008-2024")

print("\n✅ Phase 2.2 terminée")

🚀 Phase 2.2 - Naissances 2008-2024

⏭️  naissances_2008_2024 déjà extrait - SKIP

✅ Phase 2.2 terminée


### 2.3 - Décès 2008-2024 (24 MB)

In [6]:
print("🚀 Phase 2.3 - Décès 2008-2024\n")

url_deces = "https://www.insee.fr/fr/statistiques/fichier/1893253/DS_ETAT_CIVIL_DECES_COMMUNES.zip"
destination_deces = DATA_BRONZE / "deces_2008_2024"

download_and_extract_zip(url_deces, destination_deces, "Décès 2008-2024")

print("\n✅ Phase 2.3 terminée")

🚀 Phase 2.3 - Décès 2008-2024

⏭️  deces_2008_2024 déjà extrait - SKIP

✅ Phase 2.3 terminée


## 💼 Phase 3 - Économie et Emploi

### 3.1 - CSP Actifs 25-54 ans (1968-2022)

In [7]:
print("🚀 Phase 3.1 - CSP Actifs 1968-2022\n")

url_csp = "https://www.insee.fr/fr/statistiques/fichier/2837787/pop-act2554-csp-cd-6822.zip"
destination_csp = DATA_BRONZE / "csp_actifs_2554"

# Vérifier si déjà téléchargé (dossier existe déjà)
if destination_csp.exists() and any(destination_csp.iterdir()):
    print(f"⏭️  CSP Actifs déjà téléchargé - SKIP")
else:
    download_and_extract_zip(url_csp, destination_csp, "CSP Actifs 1968-2022")

print("\n✅ Phase 3.1 terminée")

🚀 Phase 3.1 - CSP Actifs 1968-2022

⏭️  CSP Actifs déjà téléchargé - SKIP

✅ Phase 3.1 terminée


### 3.2 - Secteur d'activité 1968-2022 (23.5 MB)

In [8]:
print("🚀 Phase 3.2 - Secteur d'activité 1968-2022\n")

url_secteur = "https://www.insee.fr/fr/statistiques/fichier/2837787/pop-act2554-empl-sa-sexe-cd-6822.zip"
destination_secteur = DATA_BRONZE / "secteur_activite_1968_2022"

download_and_extract_zip(url_secteur, destination_secteur, "Secteur activité 1968-2022")

print("\n✅ Phase 3.2 terminée")

🚀 Phase 3.2 - Secteur d'activité 1968-2022

📥 Secteur activité 1968-2022...
❌ Erreur: 500 Server Error:  for url: https://www.insee.fr/fr/statistiques/fichier/2837787/pop-act2554-empl-sa-sexe-cd-6822.zip

✅ Phase 3.2 terminée


### 3.3 - Comptes Communes 2000-2022 (~1-2 GB)

**⚠️ GROS FICHIERS** - Téléchargement par tranches d'années pour éviter les timeouts.

**Recommandation** : Commencer par les années récentes (2017-2022) puis télécharger le reste si nécessaire.

In [9]:
print("🚀 Phase 3.3 - Comptes Communes (par tranches)\n")

# URLs des tranches de comptes communes
comptes_tranches = [
    # Priorité 1 : Années récentes (2017-2022)
    {
        "years": "2022",
        "url": "https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/comptes-individuels-des-communes-fichier-global-a-partir-de-2000/exports/csv?limit=-1&where=exer%20%3D%202022",
        "filename": "comptes_communes_2022.csv",
        "priority": 1
    },
    {
        "years": "2021",
        "url": "https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/comptes-individuels-des-communes-fichier-global-a-partir-de-2000/exports/csv?limit=-1&where=exer%20%3D%202021",
        "filename": "comptes_communes_2021.csv",
        "priority": 1
    },
    {
        "years": "2020",
        "url": "https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/comptes-individuels-des-communes-fichier-global-a-partir-de-2000/exports/csv?limit=-1&where=exer%20%3D%202020",
        "filename": "comptes_communes_2020.csv",
        "priority": 1
    },
    {
        "years": "2019",
        "url": "https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/comptes-individuels-des-communes-fichier-global-a-partir-de-2000/exports/csv?limit=-1&where=exer%20%3D%202019",
        "filename": "comptes_communes_2019.csv",
        "priority": 1
    },
    {
        "years": "2018",
        "url": "https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/comptes-individuels-des-communes-fichier-global-a-partir-de-2000/exports/csv?limit=-1&where=exer%20%3D%202018",
        "filename": "comptes_communes_2018.csv",
        "priority": 1
    },
    {
        "years": "2017",
        "url": "https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/comptes-individuels-des-communes-fichier-global-a-partir-de-2000/exports/csv?limit=-1&where=exer%20%3D%202017",
        "filename": "comptes_communes_2017.csv",
        "priority": 1
    },
    
    # Priorité 2 : Années intermédiaires (2011-2016)
    {
        "years": "2011-2015",
        "url": "https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/comptes-individuels-des-communes-fichier-global-a-partir-de-2000/exports/csv?limit=-1&where=exer%20%3E%3D%202011%20and%20exer%20%3C%3D%202015",
        "filename": "comptes_communes_2011_2015.csv",
        "priority": 2
    },
    {
        "years": "2016",
        "url": "https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/comptes-individuels-des-communes-fichier-global-a-partir-de-2000/exports/csv?limit=-1&where=exer%20%3D%202016",
        "filename": "comptes_communes_2016.csv",
        "priority": 2
    },
    
    # Priorité 3 : Années anciennes (2000-2010) - optionnel
    {
        "years": "2000-2008",
        "url": "https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/comptes-individuels-des-communes-fichier-global-a-partir-de-2000/exports/csv?limit=-1&where=exer%20%3E%3D%202000%20and%20exer%20%3C%3D%202008",
        "filename": "comptes_communes_2000_2008.csv",
        "priority": 3
    },
    {
        "years": "2009-2010",
        "url": "https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/comptes-individuels-des-communes-fichier-global-a-partir-de-2000/exports/csv?limit=-1&where=exer%20%3E%3D%202009%20and%20exer%20%3C%3D%202010",
        "filename": "comptes_communes_2009_2010.csv",
        "priority": 3
    },
]

comptes_folder = DATA_BRONZE / "comptes_communes"
comptes_folder.mkdir(parents=True, exist_ok=True)

# Télécharger seulement priorité 1 et 2 par défaut
MAX_PRIORITY = 2  # Changer à 3 pour télécharger toutes les années

print(f"⚙️  Téléchargement des comptes communes (priorité ≤ {MAX_PRIORITY})\n")

for tranche in comptes_tranches:
    if tranche["priority"] > MAX_PRIORITY:
        print(f"⏭️  {tranche['years']} - Priorité {tranche['priority']} - SKIP (changer MAX_PRIORITY)")
        continue
    
    destination = comptes_folder / tranche["filename"]
    print(f"\n📥 Téléchargement comptes {tranche['years']}...")
    download_file(
        tranche["url"],
        destination,
        f"Comptes {tranche['years']}"
    )
    time.sleep(2)  # Pause entre les requêtes

print("\n✅ Phase 3.3 terminée")

🚀 Phase 3.3 - Comptes Communes (par tranches)

⚙️  Téléchargement des comptes communes (priorité ≤ 2)


📥 Téléchargement comptes 2022...
❌ Erreur: 404 Client Error: Not Found for url: https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/comptes-individuels-des-communes-fichier-global-a-partir-de-2000/exports/csv?limit=-1&where=exer%20%3D%202022                    

📥 Téléchargement comptes 2021...
❌ Erreur: 404 Client Error: Not Found for url: https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/comptes-individuels-des-communes-fichier-global-a-partir-de-2000/exports/csv?limit=-1&where=exer%20%3D%202021                    

📥 Téléchargement comptes 2020...
❌ Erreur: 404 Client Error: Not Found for url: https://data.economie.gouv.fr/api/explore/v2.1/catalog/datasets/comptes-individuels-des-communes-fichier-global-a-partir-de-2000/exports/csv?limit=-1&where=exer%20%3D%202020                    

📥 Téléchargement comptes 2019...
❌ Erreur: 404 Client Error: Not Found 

## 🎓 Phase 4 - Éducation et Formation

### 4.1 - Diplômes 2022 (81 MB)

In [10]:
print("🚀 Phase 4.1 - Diplômes 2022\n")

url_diplomes = "https://www.insee.fr/fr/statistiques/fichier/8581488/base-cc-diplomes-formation-2022.zip"
destination_diplomes = DATA_BRONZE / "diplomes_formation_2022"

# Vérifier si déjà téléchargé
if destination_diplomes.exists() and any(destination_diplomes.iterdir()):
    print(f"⏭️  Diplômes 2022 déjà téléchargé - SKIP")
else:
    download_and_extract_zip(url_diplomes, destination_diplomes, "Diplômes 2022")

print("\n✅ Phase 4.1 terminée")

🚀 Phase 4.1 - Diplômes 2022

⏭️  Diplômes 2022 déjà téléchargé - SKIP

✅ Phase 4.1 terminée


### 4.2 - CSP × Diplôme 1968-2022 (51.9 MB)

In [11]:
print("🚀 Phase 4.2 - CSP × Diplôme 1968-2022\n")

url_csp_diplome = "https://www.insee.fr/fr/statistiques/fichier/2837787/pop-act2554-csp-dipl-cd-6822.zip"
destination_csp_diplome = DATA_BRONZE / "csp_diplome_1968_2022"

download_and_extract_zip(url_csp_diplome, destination_csp_diplome, "CSP × Diplôme 1968-2022")

print("\n✅ Phase 4.2 terminée")

🚀 Phase 4.2 - CSP × Diplôme 1968-2022

📥 CSP × Diplôme 1968-2022...
❌ Erreur: 500 Server Error:  for url: https://www.insee.fr/fr/statistiques/fichier/2837787/pop-act2554-csp-dipl-cd-6822.zip

✅ Phase 4.2 terminée


## 📊 Récapitulatif Final

In [12]:
import os

def get_folder_size(folder):
    """Calcule la taille totale d'un dossier en GB."""
    total = 0
    for dirpath, dirnames, filenames in os.walk(folder):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if os.path.exists(fp):
                total += os.path.getsize(fp)
    return total / 1e9

print("\n" + "="*60)
print("📊 RÉCAPITULATIF FINAL")
print("="*60 + "\n")

# Calcul de la taille totale Bronze
bronze_size = get_folder_size(DATA_BRONZE)

print(f"📁 Dossier Bronze : {DATA_BRONZE}")
print(f"💾 Taille totale : {bronze_size:.2f} GB\n")

# Compter les fichiers par catégorie
categories = {
    "Référentiels COG": DATA_BRONZE / "referentiels_cog",
    "Population historique": DATA_BRONZE / "population_historique_1876_2023.xlsx",
    "Naissances": DATA_BRONZE / "naissances_2008_2024",
    "Décès": DATA_BRONZE / "deces_2008_2024",
    "CSP Actifs": DATA_BRONZE / "csp_actifs_2554",
    "Secteur activité": DATA_BRONZE / "secteur_activite_1968_2022",
    "Comptes communes": DATA_BRONZE / "comptes_communes",
    "Diplômes": DATA_BRONZE / "diplomes_formation_2022",
    "CSP × Diplôme": DATA_BRONZE / "csp_diplome_1968_2022",
}

print("📦 Données téléchargées :\n")
for name, path in categories.items():
    if path.exists():
        if path.is_dir():
            files_count = len(list(path.rglob('*')))
            size = get_folder_size(path)
            print(f"  ✅ {name:<25} {files_count:>3} fichiers - {size:.2f} GB")
        else:
            size = path.stat().st_size / 1e9
            print(f"  ✅ {name:<25} 1 fichier - {size:.2f} GB")
    else:
        print(f"  ❌ {name:<25} Non téléchargé")

print("\n" + "="*60)
print("🎉 Téléchargement terminé !")
print("="*60)
print("\n📌 Prochaines étapes :")
print("  1. Exécuter 02_exploration.ipynb pour explorer les données")
print("  2. Exécuter 03_etl.ipynb pour nettoyer et transformer (Bronze → Silver)")
print("  3. Créer les features pour le ML (Silver → Gold)")


📊 RÉCAPITULATIF FINAL

📁 Dossier Bronze : /app/data/bronze
💾 Taille totale : 2.57 GB

📦 Données téléchargées :

  ✅ Référentiels COG           22 fichiers - 0.01 GB
  ❌ Population historique     Non téléchargé
  ✅ Naissances                  2 fichiers - 0.03 GB
  ✅ Décès                       2 fichiers - 0.03 GB
  ✅ CSP Actifs                  1 fichiers - 0.03 GB
  ✅ Secteur activité            0 fichiers - 0.00 GB
  ✅ Comptes communes            0 fichiers - 0.00 GB
  ✅ Diplômes                    1 fichiers - 0.07 GB
  ✅ CSP × Diplôme               0 fichiers - 0.00 GB

🎉 Téléchargement terminé !

📌 Prochaines étapes :
  1. Exécuter 02_exploration.ipynb pour explorer les données
  2. Exécuter 03_etl.ipynb pour nettoyer et transformer (Bronze → Silver)
  3. Créer les features pour le ML (Silver → Gold)
